# 00 — Session Setup

**Run cells top-to-bottom, one at a time. Wait for each to finish before running the next.**

---
### Before running: verify your Colab Secrets

Click the **🔑 key icon** in the left sidebar. You need these secrets with a **blue toggle** (Notebook access ON):
- `ANTHROPIC_API_KEY` — from console.anthropic.com
- `HF_TOKEN` — from huggingface.co/settings/tokens
- `GITHUB_PAT` — from github.com → Settings → Developer settings → Personal access tokens

If any toggle is grey, click it to turn it blue NOW before running anything.

---
### ⚠️ Change your branch name in Cell 3 before running it

In [1]:
# ── CELL 1: Verify GPU ────────────────────────────────────────────────────────
# Expected: Tesla T4, ~15 GB VRAM
# If you see K80 or < 14 GB: Runtime → Disconnect and delete runtime → reconnect
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    icon = '✅' if vram >= 14 else '⚠️ '
    print(f'\n{icon} GPU: {name}  ({vram:.0f} GB VRAM)')
    if vram < 14:
        print('   You have a K80 (12 GB). Reconnect to get a T4 (16 GB).')
else:
    print('\n❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

Fri Sep  4 19:07:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────────
# A popup asks for permissions — click Allow on everything (normal Google behaviour).
from google.colab import drive
import os
drive.mount('/content/drive')
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted')
else:
    print('❌ Mount failed — run this cell again')

Mounted at /content/drive
✅ Drive mounted


In [4]:
# ── CELL 3: Clone or pull the GitHub repo ─────────────────────────────────────
#
# ⚠️  CHANGE BRANCH_NAME BEFORE RUNNING
#     Examples:
#       'advisor_colab_experiments'   ← advisor testing
#       'pair-1/smollm2-1.7b'         ← student pair 1
#       'main'                        ← read-only reference (do not push to main)

import os, subprocess, sys
from google.colab import userdata

BRANCH_NAME = 'btt_dk'
REPO_ORG    = 'Break-Through-Tech'
REPO_NAME   = 'Automation-Anywhere-1A-domain-specific-theme-labeling-via-slm-distillation'
REPO_DIR    = '/content/project'   # repo root

# ── Load PAT ──────────────────────────────────────────────────────────────────
try:
    PAT = userdata.get('GITHUB_PAT')
    assert PAT, 'Secret is empty'
    print(f'✅ GITHUB_PAT loaded ({len(PAT)} chars)')
except Exception as e:
    print(f'❌ GITHUB_PAT: {e}')
    print('   Open 🔑 Secrets → add GITHUB_PAT → toggle Notebook access ON')
    raise SystemExit('Cannot clone without GITHUB_PAT')

REPO_URL = f'https://{PAT}@github.com/{REPO_ORG}/{REPO_NAME}.git'

def git(args, cwd=None, check=True):
    r = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(f'❌ git error: {r.stderr.replace(PAT, "***").strip()}')
        raise RuntimeError(' '.join(args))
    return r.stdout.strip()

# ── Clone or pull ──────────────────────────────────────────────────────────────
if not os.path.exists(f'{REPO_DIR}/.git'):
    if os.path.exists(REPO_DIR):
        print('Removing broken directory ...')
        subprocess.run(['rm', '-rf', REPO_DIR])
    print(f'Cloning branch "{BRANCH_NAME}" ...')
    git(['git', 'clone', '-b', BRANCH_NAME, REPO_URL, REPO_DIR])
    print(f'✅ Cloned to {REPO_DIR}')
else:
    print(f'Pulling latest from "{BRANCH_NAME}" ...')
    git(['git', 'checkout', BRANCH_NAME], cwd=REPO_DIR)
    git(['git', 'pull', 'origin', BRANCH_NAME], cwd=REPO_DIR)
    print(f'✅ Up to date')

# ── Auto-detect where main.py lives (repo root or code/ subfolder) ────────────
if os.path.exists(f'{REPO_DIR}/main.py'):
    CODE_DIR = REPO_DIR
elif os.path.exists(f'{REPO_DIR}/code/main.py'):
    CODE_DIR = f'{REPO_DIR}/code'
else:
    CODE_DIR = None
    print('❌ Cannot find main.py — checked repo root and code/ subfolder')
    print(f'   Contents of {REPO_DIR}: {os.listdir(REPO_DIR)}')

if CODE_DIR:
    print(f'✅ Code directory: {CODE_DIR}')
    missing = [f for f in ['requirements.txt', 'requirements_colab.txt']
               if not os.path.exists(f'{CODE_DIR}/{f}')]
    if missing:
        print(f'⚠️  Missing in {CODE_DIR}: {missing}')
        print('   Push these files from your local machine, or run:')
        print('   !git -C /content/project fetch origin')
        print('   !git -C /content/project checkout origin/main -- code/requirements_colab.txt')
    else:
        print('✅ requirements.txt and requirements_colab.txt found')

    os.chdir(CODE_DIR)
    sys.path.insert(0, CODE_DIR)
    # Store CODE_DIR for other cells to use
    os.environ['SLM_CODE_DIR'] = CODE_DIR
    print(f'Working directory: {os.getcwd()}')

✅ GITHUB_PAT loaded (40 chars)
Cloning branch "btt_dk" ...
✅ Cloned to /content/project
✅ Code directory: /content/project/code
✅ requirements.txt and requirements_colab.txt found
Working directory: /content/project/code


In [5]:
# ── CELL 4: Install dependencies ──────────────────────────────────────────────
# Reads requirements files from the code directory found in Cell 3.
# Takes 3–4 minutes. Normal to see some warnings.
import os, subprocess, sys

CODE_DIR = os.environ.get('SLM_CODE_DIR', '/content/project/code')
print(f'Installing from: {CODE_DIR}')

def pip_install(filename):
    path = f'{CODE_DIR}/{filename}'
    if not os.path.exists(path):
        print(f'❌ {filename} not found at {path}')
        print('   Make sure Cell 3 ran successfully first.')
        return False
    print(f'\nInstalling {filename} ...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', path],
        capture_output=True, text=True
    )
    tail = (result.stdout + result.stderr).strip().split('\n')
    for line in tail[-5:]:
        if line.strip():
            print(f'  {line}')
    if result.returncode != 0:
        print(f'❌ pip failed for {filename}')
        return False
    print(f'✅ {filename} done')
    return True

ok1 = pip_install('requirements.txt')
ok2 = pip_install('requirements_colab.txt')
print('\n✅ All dependencies installed' if (ok1 and ok2) else '\n⚠️  Check errors above')

Installing from: /content/project/code

Installing requirements.txt ...
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 22.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.0 MB/s eta 0:00:00
✅ requirements.txt done

Installing requirements_colab.txt ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 26.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 25.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.8 MB/s eta 0:00:00
✅ requirements_colab.txt done

✅ All dependencies installe

In [6]:
# ── CELL 5: Drive folder structure and HuggingFace cache ─────────────────────
import os

DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'
for d in [f'{DRIVE_ROOT}/data/raw', f'{DRIVE_ROOT}/data/processed',
          f'{DRIVE_ROOT}/data/checkpoints', f'{DRIVE_ROOT}/outputs',
          f'{DRIVE_ROOT}/hf_cache']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
os.environ['DRIVE_ROOT'] = DRIVE_ROOT

print(f'✅ Drive folders ready under {DRIVE_ROOT}')
print(f'✅ HF model cache → {os.environ["HF_HOME"]}')

✅ Drive folders ready under /content/drive/MyDrive/slm-distillation
✅ HF model cache → /content/drive/MyDrive/slm-distillation/hf_cache


In [7]:
# ── CELL 6: Load API keys from Colab Secrets ──────────────────────────────────
import os
from google.colab import userdata

def load_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f'  ✅ {name}')
            return True
        print(f'  ⚠️  {name} is empty — {"add value in 🔑 Secrets" if required else "optional"}')
    except Exception:
        print(f'  ❌ {name} not found — {"open 🔑 Secrets and toggle Notebook access ON" if required else "optional"}')
    return not required

print('Loading secrets:')
all_ok = all([
    load_secret('ANTHROPIC_API_KEY', required=True),
    load_secret('HF_TOKEN',          required=True),
    load_secret('OPENAI_API_KEY',    required=False),
])
print('\n✅ Required secrets loaded' if all_ok else '\n❌ Fix missing secrets before running the pipeline')

Loading secrets:
  ✅ ANTHROPIC_API_KEY
  ✅ HF_TOKEN
  ❌ OPENAI_API_KEY not found — optional

✅ Required secrets loaded


In [8]:
# ── CELL 7: Verify everything is ready ───────────────────────────────────────
import os, torch

code_dir  = os.environ.get('SLM_CODE_DIR', '')
drive_ok  = os.path.exists('/content/drive/MyDrive')
gpu_ok    = torch.cuda.is_available()
vram_ok   = gpu_ok and torch.cuda.get_device_properties(0).total_memory > 14e9

checks = [
    ('T4 GPU (≥14 GB VRAM)',   vram_ok),
    ('Drive mounted',          drive_ok),
    ('Code directory found',   bool(code_dir) and os.path.exists(code_dir)),
    ('main.py present',        os.path.exists(f'{code_dir}/main.py') if code_dir else False),
    ('ANTHROPIC_API_KEY set',  'ANTHROPIC_API_KEY' in os.environ),
    ('HF_TOKEN set',           'HF_TOKEN' in os.environ),
    ('HF cache on Drive',      os.environ.get('HF_HOME','').startswith('/content/drive')),
]

print('Setup verification:')
print(f'  Code directory: {code_dir or "NOT SET"}')
print()
all_ok = True
for label, ok in checks:
    print(f'  {"✅" if ok else "❌"} {label}')
    if not ok:
        all_ok = False

print()
print('🚀 Ready! Scroll down to run the pipeline.' if all_ok else
      '⚠️  Fix ❌ items before running the pipeline.')

Setup verification:
  Code directory: /content/project/code

  ✅ T4 GPU (≥14 GB VRAM)
  ✅ Drive mounted
  ✅ Code directory found
  ✅ main.py present
  ✅ ANTHROPIC_API_KEY set
  ✅ HF_TOKEN set
  ✅ HF cache on Drive

🚀 Ready! Scroll down to run the pipeline.


---
## Run the pipeline

Choose one of the cells below. The `$SLM_CODE_DIR` variable is set by Cell 3.

# New Section

In [ ]:
# ── Full training + evaluation run (~40–60 min on T4) ─────────────────────────
import os
CODE = os.environ.get('SLM_CODE_DIR', '/content/project/code')
!python "$CODE/main.py" \
    --phase 1 \
    --config "$CODE/configs/phase1_config.yaml" \
    --device_mode colab

In [ ]:
# ── Live demo mode ────────────────────────────────────────────────────────────
import os
CODE  = os.environ.get('SLM_CODE_DIR', '/content/project/code')
DRIVE = os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/slm-distillation')

ADAPTER = f'{DRIVE}/outputs/YOUR_RUN_ID/models/lora_adapter'  # ← UPDATE

!python "$CODE/main.py" \
    --phase 1 \
    --config "$CODE/configs/phase1_config.yaml" \
    --device_mode colab \
    --mode demo \
    --adapter_dir "$ADAPTER"

In [ ]:
# ── Push code changes to GitHub ───────────────────────────────────────────────
import subprocess, os

MSG = 'describe your changes'   # ← UPDATE
REPO = '/content/project'

for cmd in [['git','add','.'], ['git','commit','-m',MSG], ['git','push']]:
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
print('Done')

---
## First session only — create your branch

Only needed if your branch does not yet exist on GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Create branch on GitHub (run ONCE) ───────────────────────────────────────
import subprocess

MY_BRANCH = 'pair-X/model-name'   # ← CHANGE THIS

for cmd in [
    ['git', 'checkout', '-b', MY_BRANCH],
    ['git', 'push', '-u', 'origin', MY_BRANCH],
]:
    r = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/project')
    print((r.stdout + r.stderr).strip())

print(f"\n✅ Branch '{MY_BRANCH}' created.")
print(f"Change BRANCH_NAME in Cell 3 to '{MY_BRANCH}' for all future sessions.")